In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

A = pd.read_csv('outputs/interm_mimic/shadow_economy_estimates.csv')
A = A.set_index('year_quarter')
SE_A = A['shadow_pct_gdp'].rename('SE_A_classical_MIMIC')

B = pd.read_csv('outputs/hybrid_estimates.csv')
B = B.set_index('year_quarter')
SE_B = B['SE_hybrid'].rename('SE_B_hybrid')

C = pd.read_csv('outputs/shadow_economy_results.csv')
C = C.set_index('year_quarter')
SE_C = C['shadow_method2'].rename('SE_C_standalone_CDA')

D = pd.read_csv('outputs/shadow_economy_electricity.csv')
D = D.set_index('quarter')
D.index.name = 'year_quarter'

SE_D = D['B1_2017_18_38.4pct_total'].rename('SE_D_electricity_KIIS')
SE_D_robust = {
    'D_A1_2010_59.49pct': D['A1_2010_59.49pct_total'],
    'D_A2_2013_53.53pct': D['A2_2013_53.53pct_total'],
    'D_B2_2018_23.8pct':  D['B2_2018_23.8pct_official'],
}

common = SE_A.index.intersection(SE_B.index).intersection(
    SE_C.index).intersection(SE_D.index)
table = pd.concat([SE_A.loc[common], SE_B.loc[common],
                   SE_C.loc[common], SE_D.loc[common]], axis=1)
table.index.name = 'year_quarter'
table.to_csv('outputs/comparison_four_approaches.csv')

print("=" * 76)
print("SUMMARY OF THE FOUR APPROACHES (% of GDP, 2010Q2-2021Q4)")
print("=" * 76)
summary = pd.DataFrame({
    col: {
        'mean':    table[col].mean(),
        'median':  table[col].median(),
        'std':     table[col].std(ddof=1),
        'min':     table[col].min(),
        'max':     table[col].max(),
        'neg_qs':  int((table[col] < 0).sum()),
    }
    for col in table.columns
}).T.round(2)
print(summary.to_string())

print("\nPairwise correlation (level):")
print(table.corr().round(3).to_string())

print("\nPairwise correlation (4q-MA, smoother):")
ma = table.rolling(4, min_periods=1).mean()
print(ma.corr().round(3).to_string())

def quarter_to_date(q):
    y, qn = q.split('Q')
    month = {'1': 2, '2': 5, '3': 8, '4': 11}[qn]
    return pd.Timestamp(year=int(y), month=month, day=15)

dates = pd.DatetimeIndex([quarter_to_date(q) for q in table.index])

fig, axes = plt.subplots(2, 1, figsize=(12, 10),
                         gridspec_kw={'height_ratios': [3, 2]})

ax = axes[0]
colors = {'SE_A_classical_MIMIC': '#1f4e79',
          'SE_B_hybrid':          '#c0504d',
          'SE_C_standalone_CDA':  '#9bbb59',
          'SE_D_electricity_KIIS':'#8064a2'}
labels = {'SE_A_classical_MIMIC': 'A. Classical MIMIC (KIIS-anchored)',
          'SE_B_hybrid':          'B. Hybrid CDA+MIMIC (Dybka et al.)',
          'SE_C_standalone_CDA':  'C. Standalone CDA (Tanzi)',
          'SE_D_electricity_KIIS':'D. Electricity (Kaufmann-Kaliberda, KIIS-anchored)'}

for col in table.columns:
    smoothed = table[col].rolling(4, min_periods=1).mean()
    ax.plot(dates, smoothed.values, color=colors[col], lw=2.2,
            label=labels[col])

ax.axvspan(pd.Timestamp(2014, 2, 1), pd.Timestamp(2015, 12, 1),
           alpha=0.07, color='red', zorder=0)
ax.axvspan(pd.Timestamp(2020, 2, 1), pd.Timestamp(2020, 9, 1),
           alpha=0.07, color='orange', zorder=0)
ax.axhline(0, color='black', lw=0.5)

ax.set_ylabel('Shadow economy (% of GDP), 4q MA', fontsize=10)
ax.set_title('Four estimates of Ukraine\'s shadow economy, 2010Q2–2021Q4',
             fontsize=12, pad=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.grid(alpha=0.3, axis='y')
ax.legend(loc='upper right', fontsize=8, frameon=True)
ax.set_xlim(dates.min(), dates.max())

ax2 = axes[1]
positions = np.arange(len(table.columns))
for i, col in enumerate(table.columns):
    s = table[col]
    ax2.plot([positions[i], positions[i]], [s.min(), s.max()],
             color=colors[col], lw=10, alpha=0.45, solid_capstyle='butt')
    ax2.plot(positions[i], s.mean(), 'o', color=colors[col], markersize=10,
             markeredgecolor='white', markeredgewidth=1.5,
             label='mean' if i == 0 else None)
    ax2.plot(positions[i], s.median(), 'D',
             color='black', markersize=7,
             label='median' if i == 0 else None)
    ax2.annotate(f'mean={s.mean():.1f}%\nmed={s.median():.1f}%',
                 (positions[i], s.max()),
                 textcoords='offset points', xytext=(15, -5),
                 fontsize=8, va='top')

ax2.axhline(0, color='black', lw=0.5)
ax2.set_xticks(positions)
ax2.set_xticklabels(['A. Classical MIMIC', 'B. Hybrid', 'C. CDA', 'D. Electricity'],
                    fontsize=9)
ax2.set_ylabel('% of GDP', fontsize=10)
ax2.set_title('Full-period range (bar = quarterly min–max, ● = mean, ◆ = median)',
              fontsize=10, pad=10)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax2.grid(alpha=0.3, axis='y')
ax2.legend(loc='upper right', fontsize=8)

plt.tight_layout()
out_path = 'outputs/comparison_four_approaches.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nSaved {out_path}')
print(f'Saved outputs/comparison_four_approaches.csv')

SUMMARY OF THE FOUR APPROACHES (% of GDP, 2010Q2-2021Q4)
                        mean  median   std    min    max  neg_qs
SE_A_classical_MIMIC   41.16   39.88  5.00  34.59  53.24     0.0
SE_B_hybrid             4.00    3.38  2.13  -1.07  11.52     1.0
SE_C_standalone_CDA     4.00    3.56  2.13   0.00   8.92     0.0
SE_D_electricity_KIIS  37.94   39.64  5.37  23.29  48.13     0.0

Pairwise correlation (level):
                       SE_A_classical_MIMIC  SE_B_hybrid  SE_C_standalone_CDA  SE_D_electricity_KIIS
SE_A_classical_MIMIC                  1.000        0.356               -0.306                 -0.040
SE_B_hybrid                           0.356        1.000                0.002                  0.041
SE_C_standalone_CDA                  -0.306        0.002                1.000                  0.422
SE_D_electricity_KIIS                -0.040        0.041                0.422                  1.000

Pairwise correlation (4q-MA, smoother):
                       SE_A_classical_MIM